In [1]:
import nltk
nltk.download('treebank')

[nltk_data] Downloading package treebank to
[nltk_data]     /Users/priyakeshri/nltk_data...
[nltk_data]   Package treebank is already up-to-date!


True

In [2]:
import torch
from torch.nn.utils.rnn import pad_sequence
from sklearn.model_selection import train_test_split
from nltk.corpus import treebank

# ---------------- LOAD DATA ----------------
sentences = treebank.tagged_sents()
sentences = [[(w.lower(), t) for w, t in sent] for sent in sentences]

# ---------------- BUILD VOCAB ----------------
word_set = set()
tag_set = set()

for sent in sentences:
    for w, t in sent:
        word_set.add(w)
        tag_set.add(t)

# Add special tokens
word2idx = {w: i+2 for i, w in enumerate(word_set)}
word2idx["<PAD>"] = 0
word2idx["<UNK>"] = 1

tag2idx = {t: i for i, t in enumerate(tag_set)}
idx2tag = {i: t for t, i in tag2idx.items()}

print("Vocab size:", len(word2idx))
print("Tag size:", len(tag2idx))


# ---------------- ENCODING ----------------
def encode_sentence(sentence):
    words = [word2idx.get(w, word2idx["<UNK>"]) for w, t in sentence]
    tags = [tag2idx[t] for w, t in sentence]
    return torch.tensor(words), torch.tensor(tags)


encoded_data = [encode_sentence(s) for s in sentences]

# ---------------- SPLIT ----------------
train_data, test_data = train_test_split(encoded_data, test_size=0.2, random_state=42)


# ---------------- PAD SEQUENCES ----------------
def collate_fn(batch):
    words = [item[0] for item in batch]
    tags = [item[1] for item in batch]

    words_padded = pad_sequence(words, batch_first=True, padding_value=0)
    tags_padded = pad_sequence(tags, batch_first=True, padding_value=-1)

    return words_padded, tags_padded


print("\n--- Assignment 1 Output ---")
print("Sample encoded sentence:", encoded_data[0])

Vocab size: 11389
Tag size: 46

--- Assignment 1 Output ---
Sample encoded sentence: (tensor([ 4409,   855,  8738,   832,   438,  6273,  8738,  6058,   168,  8275,
         8468,  8467, 10431,  6858,  3311,  7334,   656,  8442]), tensor([34, 34,  0, 24, 13, 43,  0, 22, 37, 29, 40, 20, 29, 43, 40, 34, 24, 11]))


In [3]:
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

# ---------------- MODEL ----------------
class POS_LSTM(nn.Module):
    def __init__(self, vocab_size, tagset_size, embedding_dim=128, hidden_dim=128):
        super().__init__()
        
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, tagset_size)

    def forward(self, x):
        x = self.embedding(x)
        x, _ = self.lstm(x)
        x = self.fc(x)
        return x


# ---------------- INIT ----------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = POS_LSTM(len(word2idx), len(tag2idx)).to(device)
criterion = nn.CrossEntropyLoss(ignore_index=-1)
optimizer = optim.Adam(model.parameters(), lr=0.001)

train_loader = DataLoader(train_data, batch_size=32, shuffle=True, collate_fn=collate_fn)


# ---------------- TRAIN ----------------
EPOCHS = 5

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for words, tags in train_loader:
        words, tags = words.to(device), tags.to(device)

        optimizer.zero_grad()
        outputs = model(words)

        outputs = outputs.view(-1, len(tag2idx))
        tags = tags.view(-1)

        loss = criterion(outputs, tags)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")


# ---------------- PREDICTION ----------------
def predict(sentence):
    model.eval()
    
    words = [word2idx.get(w.lower(), word2idx["<UNK>"]) for w in sentence]
    words = torch.tensor(words).unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(words)
        preds = torch.argmax(outputs, dim=2).squeeze().cpu().numpy()

    return [idx2tag[p] for p in preds]


# Example
test_sentence = ["the", "market", "fell", "today"]
pred_tags = predict(test_sentence)

print("\n--- Assignment 2 Output ---")
print(list(zip(test_sentence, pred_tags)))

Epoch 1, Loss: 225.7230
Epoch 2, Loss: 111.0310
Epoch 3, Loss: 78.7614
Epoch 4, Loss: 60.7525
Epoch 5, Loss: 48.5979

--- Assignment 2 Output ---
[('the', 'DT'), ('market', 'NN'), ('fell', 'VBD'), ('today', 'NN')]


In [4]:
# ---------------- BASELINE ----------------
from collections import Counter

word_tag_freq = {}

for sent in sentences:
    for w, t in sent:
        if w not in word_tag_freq:
            word_tag_freq[w] = Counter()
        word_tag_freq[w][t] += 1

baseline = {w: tags.most_common(1)[0][0] for w, tags in word_tag_freq.items()}


def baseline_predict(sentence):
    return [baseline.get(w.lower(), "NN") for w in sentence]


# ---------------- EVALUATION ----------------
def evaluate_model(test_data):
    model.eval()

    correct = 0
    total = 0

    for words, tags in test_data:
        words = words.unsqueeze(0).to(device)

        with torch.no_grad():
            outputs = model(words)
            preds = torch.argmax(outputs, dim=2).squeeze().cpu()

        for p, t in zip(preds, tags):
            if t.item() == -1:
                continue
            if p.item() == t.item():
                correct += 1
            total += 1

    return correct / total


def evaluate_baseline(test_data):
    correct = 0
    total = 0

    for words, tags in test_data:
        words_list = words.tolist()
        words_text = [list(word2idx.keys())[list(word2idx.values()).index(w)] if w in word2idx.values() else "<UNK>" for w in words_list]

        preds = baseline_predict(words_text)

        for p, t in zip(preds, tags):
            if t.item() == -1:
                continue
            if tag2idx[p] == t.item():
                correct += 1
            total += 1

    return correct / total


# ---------------- RUN ----------------
acc_model = evaluate_model(test_data)
acc_baseline = evaluate_baseline(test_data)

print("\n--- Assignment 3 Output ---")
print(f"LSTM Accuracy: {acc_model:.4f}")
print(f"Baseline Accuracy: {acc_baseline:.4f}")

TypeError: iteration over a 0-d tensor